# NC-SLU: NC-SSM-Bi Training + NC-OPAL (EMNLP 2026)

**Cell 0** Setup — Drive mount, FSC zip 자동 탐색/추출, Repo clone (~3 min)
**Cell 1** Data — FSC 오디오 로딩, 25 base + 6 new intent split (~3 min)
**Cell 2** Functions — train / eval / NC-OPAL 함수 정의 (instant)
**Cell 3** Train — NC-SSM-Bi 50ep 학습 + Local/Drive 저장 + reload 검증 (~16h on T4)
**Cell 4** NC-OPAL — 4개 backbone checkpoint → NC-OPAL → Table 1 LaTeX (~20 min)

**준비물:** Google Drive에 `fluentai.zip` (FSC 데이터셋) 업로드, GPU 런타임 (T4+)

In [ ]:
# ================================================================
# Cell 0: SETUP — Drive mount + FSC 추출 + Repo clone
# ================================================================
import os, sys, subprocess, time, glob, zipfile, shutil

# ---- 1. Google Drive mount ----
from google.colab import drive
try:
    drive.mount('/content/drive')
except ValueError:
    print('[OK] Drive already mounted')

DRIVE_DIR = '/content/drive/MyDrive/NC-SLU-checkpoints'
os.makedirs(DRIVE_DIR, exist_ok=True)

# ---- 2. FSC 데이터셋 찾기 & 추출 ----
def find_fsc_extracted():
    """이미 /content 아래에 추출된 FSC가 있는지 확인"""
    hits = [f for f in glob.glob('/content/**/train_data.csv', recursive=True)
            if 'drive' not in f]
    if hits:
        return os.path.dirname(os.path.dirname(hits[0]))
    return None

def find_fsc_zip():
    """Drive에서 FSC zip 파일 탐색 (3단계)"""
    # (a) 자주 쓰는 이름 직접 확인
    for name in ['fluentai.zip', 'fsc.zip', 'fluent_speech_commands_dataset.zip']:
        p = f'/content/drive/MyDrive/{name}'
        if os.path.isfile(p):
            print(f'[FOUND] {p}')
            return p

    # (b) Drive 전체 검색 (depth 4)
    print('[SEARCH] Drive 전체에서 FSC zip 검색 중...')
    r = subprocess.run(
        ['find', '/content/drive/MyDrive', '-maxdepth', '4',
         '-iname', '*.zip', '-type', 'f'],
        capture_output=True, text=True, timeout=180)
    for line in r.stdout.strip().split('\n'):
        line = line.strip()
        if not line:
            continue
        base = os.path.basename(line).lower()
        if any(kw in base for kw in ['fluent', 'fsc']):
            print(f'[FOUND] {line}')
            return line

    # (c) 못 찾으면 수동 업로드
    print('[!] Drive에서 FSC zip을 찾지 못했습니다. 파일 업로드 대화상자를 엽니다...')
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('FSC zip 파일이 필요합니다!')
    fname = list(uploaded.keys())[0]
    local = f'/content/{fname}'
    # Drive에 캐시
    shutil.copy(local, f'/content/drive/MyDrive/{fname}')
    print(f'[CACHED] /content/drive/MyDrive/{fname}')
    return local

FSC_DIR = find_fsc_extracted()
if FSC_DIR:
    print(f'[OK] FSC already at {FSC_DIR}')
else:
    zip_path = find_fsc_zip()
    sz = os.path.getsize(zip_path) / 1e9
    print(f'Extracting {os.path.basename(zip_path)} ({sz:.1f} GB)...')
    t0 = time.time()
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall('/content')
    print(f'  Done in {time.time()-t0:.0f}s')
    FSC_DIR = find_fsc_extracted()
    assert FSC_DIR, 'train_data.csv not found after extraction!'

assert os.path.isfile(f'{FSC_DIR}/data/train_data.csv'), f'Bad FSC_DIR: {FSC_DIR}'
print(f'[OK] FSC_DIR = {FSC_DIR}')

# ---- 3. Repo clone ----
REPO = '/content/NC-KWS-FineTuning'
os.chdir('/content')
if os.path.exists(REPO):
    subprocess.run(['rm', '-rf', REPO])
subprocess.run(['git', 'clone',
    'https://github.com/DrJinHoChoi/NC-KWS-FineTuning.git', REPO], check=True)

LOCAL_DIR = f'{REPO}/results/models'
os.makedirs(LOCAL_DIR, exist_ok=True)
sys.path.insert(0, REPO) if REPO not in sys.path else None
os.chdir(REPO)

assert os.path.isfile(f'{REPO}/nanomamba.py'), 'nanomamba.py missing!'
print(f'[OK] REPO      = {REPO}')
print(f'[OK] LOCAL_DIR = {LOCAL_DIR}')
print(f'[OK] DRIVE_DIR = {DRIVE_DIR}')

In [ ]:
# ================================================================
# Cell 1: DATA LOADING
#   - Load FSC train/val/test metadata + audio
#   - Split: 25 base + 6 new intents (seed 42, fixed)
#   - Build data dicts with BASE_INTENTS order preserved
# ================================================================
import time, copy, random, json, shutil
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import soundfile as sf
from nanomamba import (create_nc_tcn_20k, create_nc_tcn_20k_bi,
                       create_nanomamba_nc_20k, create_nanomamba_nc_20k_bi)

SR = 16000
MAX_LEN = SR * 3  # 3 seconds
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# --- NC-OPAL v2 hyperparameters ---
LORA_RANK = 2
LORA_ALPHA = 4
LAMBDA_KD = 5.0
KD_TEMP = 3.0
REHEARSAL_PER_CLASS = 30
OLD_AUG_FACTOR = 4
NEW_AUG_FACTOR = 8
STAGE1_EPOCHS = 5
STAGE2_EPOCHS = 20
STAGE1_LR = 4.5e-3
STAGE2_LR = 1.5e-3
BATCH_SIZE = 32
GRAD_CLIP = 1.0
BASE_EPOCHS = 50

# --- FSC loaders ---
def load_fsc_split(split):
    df = pd.read_csv(f'{FSC_DIR}/data/{split}_data.csv')
    return [{'intent': f\"{r['action']}_{r['object']}_{r['location']}\",
             'path': f'{FSC_DIR}/{r["path"]}'} for _, r in df.iterrows()]

def load_audio(path):
    a, sr = sf.read(path, dtype='float32')
    if a.ndim > 1: a = a[:, 0]
    if sr != SR:
        ratio = SR / sr
        n = int(len(a) * ratio)
        idx = np.linspace(0, len(a)-1, n).astype(np.float32)
        f_ = np.floor(idx).astype(int)
        c_ = np.minimum(f_ + 1, len(a)-1)
        a = a[f_] * (1 - (idx - f_)) + a[c_] * (idx - f_)
    if len(a) > MAX_LEN: a = a[:MAX_LEN]
    elif len(a) < MAX_LEN: a = np.pad(a, (0, MAX_LEN - len(a)))
    return a.astype(np.float32)

def build_dataset(data_list, intents):
    result = defaultdict(list)
    for d in data_list:
        if d['intent'] in intents:
            try: result[d['intent']].append(load_audio(d['path']))
            except Exception: pass
    # CRITICAL: preserve intents order for label consistency
    return {k: result[k] for k in intents if k in result}

# --- Load metadata ---
print('Loading FSC metadata...')
train_data = load_fsc_split('train')
val_data   = load_fsc_split('valid')
test_data  = load_fsc_split('test')

all_intents = sorted(set(d['intent'] for d in train_data))
print(f'  Unique intents: {len(all_intents)}')

# --- Intent split (fixed seed 42) ---
random.seed(42)
intents_shuffled = all_intents.copy()
random.shuffle(intents_shuffled)
BASE_INTENTS = intents_shuffled[:25]
NEW_INTENTS  = intents_shuffled[25:]
N_SHOT = 20
print(f'  Base: {len(BASE_INTENTS)}, New: {len(NEW_INTENTS)}')

# --- Load audio ---
print('Loading audio (may take a few minutes)...')
t0 = time.time()
base_train     = build_dataset(train_data, BASE_INTENTS)
base_val       = build_dataset(val_data,   BASE_INTENTS)
base_test      = build_dataset(test_data,  BASE_INTENTS)
new_all_train  = build_dataset(train_data, NEW_INTENTS)
new_train      = {k: v[:N_SHOT] for k, v in new_all_train.items()}
new_test       = build_dataset(test_data,  NEW_INTENTS)
base_train_sub = {k: v[:REHEARSAL_PER_CLASS] for k, v in base_train.items()}
all_test_data  = {**base_test, **new_test}
print(f'  Loaded in {time.time()-t0:.1f}s')

# --- Sanity: intent order must match ---
assert list(base_train.keys()) == BASE_INTENTS, 'base_train order mismatch!'
assert list(base_test.keys())  == BASE_INTENTS, 'base_test order mismatch!'
print('[OK] All data loaded, intent order verified')

In [ ]:
# ================================================================
# Cell 2: FUNCTION DEFINITIONS
#   - augment: time stretch, shift, gain, masking, noise
#   - LoRALinear: low-rank adapter (r=2, alpha=4)
#   - train_base_slu: train base model with augmentation
#   - evaluate_slu: per-intent accuracy evaluation
#   - ncopal_slu: NC-OPAL v2 (proto init + LoRA + KD)
# ================================================================

def aug_time_stretch(a, rate):
    n = int(len(a) / rate)
    idx = np.linspace(0, len(a)-1, n).astype(np.float32)
    f_ = np.floor(idx).astype(int)
    c_ = np.minimum(f_ + 1, len(a)-1)
    out = a[f_] * (1 - (idx - f_)) + a[c_] * (idx - f_)
    if len(out) > MAX_LEN: out = out[:MAX_LEN]
    elif len(out) < MAX_LEN: out = np.pad(out, (0, MAX_LEN - len(out)))
    return out.astype(np.float32)

def augment(a):
    out = a.copy()
    if np.random.random() < 0.5: out = aug_time_stretch(out, np.random.uniform(0.9, 1.1))
    if np.random.random() < 0.5: out = np.roll(out, np.random.randint(-4800, 4800))
    if np.random.random() < 0.5: out = out * (10 ** (np.random.uniform(-4, 4) / 20))
    if np.random.random() < 0.3:
        ml = int(len(out) * np.random.uniform(0.05, 0.1))
        s = np.random.randint(0, len(out) - ml)
        out[s:s+ml] = 0.0
    out = out + np.random.randn(len(out)).astype(np.float32) * 0.003
    return out.astype(np.float32)

class LoRALinear(nn.Module):
    def __init__(self, original, rank=LORA_RANK, alpha=LORA_ALPHA):
        super().__init__()
        self.original = original
        self.scaling = alpha / rank
        self.lora_A = nn.Parameter(torch.randn(original.in_features, rank) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(rank, original.out_features))
    def forward(self, x):
        return self.original(x) + (x @ self.lora_A @ self.lora_B) * self.scaling

def label_smooth_ce(logits, targets, smoothing=0.1):
    lp = F.log_softmax(logits, dim=-1)
    nll = -lp.gather(dim=-1, index=targets.unsqueeze(1)).squeeze(1)
    return ((1 - smoothing) * nll + smoothing * (-lp.mean(dim=-1))).mean()

def train_base_slu(intents, train_data, val_data, epochs=BASE_EPOCHS, model_fn=create_nc_tcn_20k):
    n_cls = len(intents)
    model = model_fn(n_classes=n_cls).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    print(f'  Base model: {n_cls} intents, {n_params:,} params, {epochs} epochs')
    X, Y = [], []
    for i, intent in enumerate(intents):
        for a in train_data.get(intent, []):
            X.append(a); Y.append(i)
    opt = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    for ep in range(epochs):
        model.train()
        perm = np.random.permutation(len(X))
        eloss, nb = 0, 0
        for i in range(0, len(X), BATCH_SIZE):
            bi = perm[i:i+BATCH_SIZE]
            batch = [augment(X[j]) if np.random.random() < 0.5 else X[j] for j in bi]
            bx = torch.stack([torch.from_numpy(a).float() for a in batch]).to(DEVICE)
            by = torch.tensor([Y[j] for j in bi], dtype=torch.long).to(DEVICE)
            loss = F.cross_entropy(model(bx), by)
            opt.zero_grad(); loss.backward(); opt.step()
            eloss += loss.item(); nb += 1
        sched.step()
        if (ep+1) % 10 == 0 or ep == 0:
            model.eval()
            vc, vt = 0, 0
            with torch.no_grad():
                for i, intent in enumerate(intents):
                    for a in val_data.get(intent, []):
                        x = torch.from_numpy(a).float().unsqueeze(0).to(DEVICE)
                        vt += 1; vc += (model(x).argmax(-1).item() == i)
            print(f'    Ep {ep+1}/{epochs} loss={eloss/nb:.4f} val={vc/max(vt,1)*100:.1f}%')
    return model

def evaluate_slu(model, intents, test_data):
    model.eval()
    correct, total = 0, 0
    per_intent = {}
    with torch.no_grad():
        for i, intent in enumerate(intents):
            ic, it = 0, 0
            for a in test_data.get(intent, []):
                x = torch.from_numpy(a).float().unsqueeze(0).to(DEVICE)
                it += 1; ic += (model(x).argmax(-1).item() == i)
            per_intent[intent] = ic / max(it, 1)
            correct += ic; total += it
    return correct / max(total, 1), per_intent

def ncopal_slu(base_model, base_intents, new_intents, new_train, base_train_sub):
    """NC-OPAL v2: prototype init + LoRA(r=2) + KD(lambda=5)"""
    model = copy.deepcopy(base_model)
    n_old, n_new = len(base_intents), len(new_intents)
    n_total = n_old + n_new
    d = model.classifier.in_features

    # --- Embedding hook ---
    emb_hook = {}
    def hook_fn(m, inp, out): emb_hook['e'] = inp[0].detach()
    def get_emb(a):
        h = model.classifier.register_forward_hook(hook_fn)
        with torch.no_grad(): model(torch.from_numpy(a).float().unsqueeze(0).to(DEVICE))
        h.remove()
        return emb_hook['e'].squeeze(0)

    # --- Prototype-initialized head ---
    old_head = model.classifier
    new_head = nn.Linear(d, n_total).to(DEVICE)
    with torch.no_grad():
        new_head.weight[:n_old] = old_head.weight
        new_head.bias[:n_old] = old_head.bias
        scale = old_head.weight.norm(dim=1).mean().item()
        for i, intent in enumerate(new_intents):
            embs = [get_emb(a) for a in new_train.get(intent, [])]
            if embs:
                proto = F.normalize(torch.stack(embs).mean(0), dim=0)
                new_head.weight[n_old+i] = proto * scale
                new_head.bias[n_old+i] = old_head.bias.mean().item()
    model.classifier = new_head

    # --- LoRA injection ---
    loras = []
    for block in model.blocks:
        for attr in ['in_proj', 'out_proj']:
            if hasattr(block, attr):
                orig = getattr(block, attr)
                if isinstance(orig, nn.Linear):
                    lo = LoRALinear(orig).to(DEVICE)
                    setattr(block, attr, lo); loras.append(lo)

    # --- Teacher for KD ---
    teacher = copy.deepcopy(base_model).to(DEVICE)
    teacher.eval()
    for p in teacher.parameters(): p.requires_grad = False

    # --- Training data (augmented) ---
    X, Y = [], []
    for i, intent in enumerate(new_intents):
        li = n_old + i
        for a in new_train.get(intent, []):
            X.append(a); Y.append(li)
            for _ in range(NEW_AUG_FACTOR):
                X.append(augment(a)); Y.append(li)
    for i, intent in enumerate(base_intents):
        for a in base_train_sub.get(intent, [])[:REHEARSAL_PER_CLASS]:
            X.append(a); Y.append(i)
            for _ in range(OLD_AUG_FACTOR):
                X.append(augment(a)); Y.append(i)
    print(f'  NC-OPAL train: {len(X)} samples')

    # --- Stage 1: head only ---
    for p in model.parameters(): p.requires_grad = False
    for p in new_head.parameters(): p.requires_grad = True
    opt1 = torch.optim.AdamW(new_head.parameters(), lr=STAGE1_LR, weight_decay=0.01)
    model.train()
    for ep in range(STAGE1_EPOCHS):
        perm = np.random.permutation(len(X))
        for i in range(0, len(X), BATCH_SIZE):
            bi = perm[i:i+BATCH_SIZE]
            bx = torch.stack([torch.from_numpy(X[j]).float() for j in bi]).to(DEVICE)
            by = torch.tensor([Y[j] for j in bi], dtype=torch.long).to(DEVICE)
            loss = label_smooth_ce(model(bx), by, 0.1)
            opt1.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(new_head.parameters(), GRAD_CLIP)
            opt1.step()

    # --- Stage 2: LoRA + head + KD ---
    for m in loras:
        for p in m.parameters(): p.requires_grad = True
    opt2 = torch.optim.AdamW([
        {'params': [p for m in loras for p in m.parameters()], 'lr': STAGE2_LR},
        {'params': new_head.parameters(), 'lr': STAGE2_LR * 0.5},
    ], weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=STAGE2_EPOCHS)
    for ep in range(STAGE2_EPOCHS):
        perm = np.random.permutation(len(X))
        for i in range(0, len(X), BATCH_SIZE):
            bi = perm[i:i+BATCH_SIZE]
            bx = torch.stack([torch.from_numpy(X[j]).float() for j in bi]).to(DEVICE)
            by = torch.tensor([Y[j] for j in bi], dtype=torch.long).to(DEVICE)
            logits = model(bx)
            cls_loss = label_smooth_ce(logits, by, 0.05)
            kd_loss = torch.tensor(0.0).to(DEVICE)
            old_mask = by < n_old
            if old_mask.any():
                with torch.no_grad(): t_logits = teacher(bx[old_mask])
                s_base = logits[old_mask][:, :n_old] / KD_TEMP
                t_base = t_logits / KD_TEMP
                kd_loss = F.kl_div(F.log_softmax(s_base, -1),
                                   F.softmax(t_base, -1),
                                   reduction='batchmean') * (KD_TEMP**2)
            total = cls_loss + LAMBDA_KD * min(1.0, ep / 5.0) * kd_loss
            opt2.zero_grad(); total.backward()
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
            opt2.step()
        sched.step()
    return model, base_intents + new_intents

print('[OK] All functions defined')

In [ ]:
# ================================================================
# Cell 3: TRAIN NC-SSM-Bi (50 epochs)
#   - Train from scratch on 25 base intents
#   - Save checkpoint to Local + Drive
#   - Reload from disk and verify accuracy matches
#   - Expected: ~16 hours on T4, ~25K params, ~95% base test acc
# ================================================================

name = 'NC-SSM-Bi'
fn = create_nanomamba_nc_20k_bi

print(f'{"="*60}')
print(f'  {name} - {BASE_EPOCHS} epochs')
print(f'{"="*60}')

t0 = time.time()
model_ssmbi = train_base_slu(BASE_INTENTS, base_train, base_val,
                              epochs=BASE_EPOCHS, model_fn=fn)
dt = time.time() - t0

# --- Evaluate ---
acc, _ = evaluate_slu(model_ssmbi, BASE_INTENTS, base_test)
n_params = sum(p.numel() for p in model_ssmbi.parameters())
print(f'\n  Params: {n_params:,}')
print(f'  Base test acc: {acc*100:.2f}%')
print(f'  Time: {dt/3600:.2f}h')

# --- Save to Local + Drive (CPU state_dict) ---
state = {k: v.detach().cpu() for k, v in model_ssmbi.state_dict().items()}
fname = f'{name}_base_{BASE_EPOCHS}ep.pt'
local_path = f'{LOCAL_DIR}/{fname}'
drive_path = f'{DRIVE_DIR}/{fname}'
torch.save(state, local_path)
shutil.copy(local_path, drive_path)

meta = {'backbone': name, 'params': n_params,
        'base_acc': float(acc), 'time_s': dt,
        'epochs': BASE_EPOCHS, 'intents': BASE_INTENTS}
meta_path = local_path.replace('.pt', '_meta.json')
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)
shutil.copy(meta_path, drive_path.replace('.pt', '_meta.json'))
print(f'  [Saved] {local_path}')
print(f'  [Saved] {drive_path}')

# --- Reload and verify ---
s2 = torch.load(local_path, map_location='cpu')
m2 = fn(n_classes=len(BASE_INTENTS))
m2.load_state_dict(s2)
m2.to(DEVICE).eval()
acc2, _ = evaluate_slu(m2, BASE_INTENTS, base_test)
print(f'  [Verify] reload acc: {acc2*100:.2f}% (expected {acc*100:.2f}%)')
assert abs(acc - acc2) < 0.01, 'CHECKPOINT CORRUPTED'
print(f'  [OK] {name} done')

In [ ]:
# ================================================================
# Cell 4: NC-OPAL ON ALL 4 BACKBONES
#   - Restore NC-TCN, NC-TCN-Bi, NC-SSM checkpoints from Drive
#   - NC-SSM-Bi from Cell 3 (already in LOCAL_DIR)
#   - Run NC-OPAL on each: prototype init + LoRA + KD
#   - Output: Table 1 LaTeX rows for EMNLP paper
# ================================================================

BACKBONES = [
    ('NC-TCN',    create_nc_tcn_20k),
    ('NC-TCN-Bi', create_nc_tcn_20k_bi),
    ('NC-SSM',    create_nanomamba_nc_20k),
    ('NC-SSM-Bi', create_nanomamba_nc_20k_bi),
]

# --- Restore checkpoints from Drive if not local ---
print('Checking checkpoints...')
for n, _ in BACKBONES:
    lp = f'{LOCAL_DIR}/{n}_base_{BASE_EPOCHS}ep.pt'
    dp = f'{DRIVE_DIR}/{n}_base_{BASE_EPOCHS}ep.pt'
    if not os.path.exists(lp) and os.path.exists(dp):
        shutil.copy(dp, lp)
        print(f'  [restored] {n} from Drive')
    elif os.path.exists(lp):
        print(f'  [local]    {n}')
    else:
        print(f'  [MISSING]  {n} -- will be skipped')

# --- Run NC-OPAL on each ---
results_opal = {}

for name, fn in BACKBONES:
    path = f'{LOCAL_DIR}/{name}_base_{BASE_EPOCHS}ep.pt'
    if not os.path.exists(path):
        print(f'\n[SKIP] {name}: no checkpoint')
        continue

    print(f'\n{"="*60}')
    print(f'  NC-OPAL: {name}')
    print(f'{"="*60}')

    # Fresh load
    state = torch.load(path, map_location='cpu')
    model = fn(n_classes=len(BASE_INTENTS))
    model.load_state_dict(state)
    model.to(DEVICE).eval()
    n_params = sum(p.numel() for p in model.parameters())

    # Base accuracy sanity check
    base_acc, _ = evaluate_slu(model, BASE_INTENTS, base_test)
    print(f'  Params: {n_params:,}  Base acc: {base_acc*100:.2f}%')
    if base_acc < 0.5:
        print(f'  [WARN] base_acc too low -- check intent order!')

    # NC-OPAL
    t0 = time.time()
    opal_m, opal_intents = ncopal_slu(
        model, BASE_INTENTS, NEW_INTENTS, new_train, base_train_sub)
    acc_all, per = evaluate_slu(opal_m, opal_intents, all_test_data)
    acc_base = float(np.mean([per[i] for i in BASE_INTENTS]))
    acc_new  = float(np.mean([per[i] for i in NEW_INTENTS]))
    fgt = base_acc - acc_base
    dt = time.time() - t0

    print(f'  [{dt:.0f}s] All={acc_all*100:.1f}%  Base={acc_base*100:.1f}%  '
          f'New={acc_new*100:.1f}%  Fgt={fgt*100:+.1f}%p')

    results_opal[name] = {
        'params': n_params,
        'base_pretrain': float(base_acc),
        'all': float(acc_all), 'base': float(acc_base),
        'new': float(acc_new), 'fgt': float(fgt),
    }

# --- Save results ---
out_path = f'{REPO}/results/exp1_{BASE_EPOCHS}ep_all4.json'
with open(out_path, 'w') as f:
    json.dump(results_opal, f, indent=2)
shutil.copy(out_path, f'{DRIVE_DIR}/exp1_{BASE_EPOCHS}ep_all4.json')

# --- Table 1 LaTeX rows ---
print(f'\n{"="*60}')
print(f'  TABLE 1 (EMNLP 2026)')
print(f'{"="*60}')
print(f'{"Backbone":<12} {"Params":>6} {"All":>6} {"Base":>6} {"New":>6} {"Fgt":>6}')
print('-' * 48)
for name, r in results_opal.items():
    pk = f'{r["params"]/1000:.0f}K'
    print(f'{name:<12} {pk:>6} {r["all"]*100:>5.1f}% {r["base"]*100:>5.1f}% '
          f'{r["new"]*100:>5.1f}% {r["fgt"]*100:>+5.1f}%')
print()
print('LaTeX:')
for name, r in results_opal.items():
    pk = f'{r["params"]/1000:.0f}K'
    print(f'{name:<12} & {pk} & {r["all"]*100:.1f} & {r["base"]*100:.1f} '
          f'& {r["new"]*100:.1f} & $-${abs(r["fgt"])*100:.1f} \\\\')